# Valutazione del retrieval

Si valuta il retrieval **a partire dalla query Cypher**; Due esperimenti; i casi sono in `data/valutazione/suite_valutazione.json`.

- **E1 — ricerca semantica sulle descrizioni dei nodi** (MiniLM): 24 domande. Per 21 la risposta giusta (insieme pertinente) si calcola con una query Cypher esatta; per 3 non esiste e si giudica a mano. Quattro strategie: **solo semantica**; **indice prima** e poi filtri del grafo (50, 100 o 500 candidati); **grafo prima** e poi ricerca semantica; **grafo prima (domanda intera)**.
- **E2 — ricerca di immagini** (CLIP): a) riconoscimento della persona; b) concetti visivi con etichetta ricavata dal grafo; c) concetti visivi giudicati a mano; d) immagine della scheda giocatore (quella con labelconfidence più alta).


In [1]:
# [Cella 1] Setup: codice, connessione, casi della suite, modelli
import importlib
import json
import random
import sys
import time
from getpass import getpass
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
import wc_retrieval.embedding
import wc_retrieval.grafo
import wc_retrieval.ricerca
import wc_retrieval.valutazione
for modulo in (wc_retrieval.grafo, wc_retrieval.ricerca, wc_retrieval.embedding, wc_retrieval.valutazione):
    importlib.reload(modulo)
from wc_retrieval import embedding as emb
from wc_retrieval import valutazione as val
from wc_retrieval.grafo import BASE, Grafo

VAL = ROOT / "data" / "valutazione"
RIS = VAL / "risultati"
RIS.mkdir(parents=True, exist_ok=True)
# I casi della valutazione: domande di E1 e concetti di E2
suite = json.load(open(VAL / "suite_valutazione.json", encoding="utf-8"))
E1, E2 = suite["e1"], suite["e2"]

g = Grafo(getpass("Password Neo4j: "))
# I due modelli: MiniLM per le domande sui nodi, CLIP per quelle sulle immagini
minilm = emb.carica_modello_testo()
clip = emb.carica_clip()
# Testo → vettore, come lista di float (il formato del parametro Cypher)
frase = lambda modello, testo: [float(x) for x in emb.vettori_testo(modello, [testo])[0]]
print("domande E1:", len(E1), "| giocatori E2a:", len(E2["a_persona"]["famosi"]) + len(E2["a_persona"]["meno_noti"]),
      "| concetti E2b:", len(E2["b_concetti"]), "| E2c:", len(E2["c_giudizi"]), "| E2d:", len(E2["d_scheda"]["giocatori"]))

domande E1: 24 | giocatori E2a: 30 | concetti E2b: 5 | E2c: 5 | E2d: 20


## E1 — Insiemi pertinenti
Per le domande con risposta esatta, la query Cypher della suite dà l'insieme pertinente. Il numero di nodi trovati deve coincidere con quello atteso, calcolato da Claude sulle tabelle di `data/processed`: è il controllo che le query esatte siano scritte giuste.

In [2]:
# [Cella 2] E1: insiemi pertinenti dalle query esatte, confrontati con le dimensioni attese
pertinenti = {}
righe = []
# Per ogni domanda con risposta esatta, la query Cypher della suite dà l'insieme dei nodi giusti
for d in E1:
    if d["pertinenti"]:
        pertinenti[d["id"]] = set(g.leggi(d["pertinenti"])["uri"])
        righe.append({"id": d["id"], "caso": d["caso"], "classe": d["classe"], "attesi": d["attesi"],
                      "trovati": len(pertinenti[d["id"]]), "ok": len(pertinenti[d["id"]]) == d["attesi"]})
controllo = pd.DataFrame(righe)
print(controllo.to_string(index=False))
print(f"insiemi pertinenti con la dimensione attesa: {controllo['ok'].sum()}/{len(controllo)}")

id  caso        classe  attesi  trovati   ok
P1     1        Player       3        3 True
P2     1        Player       6        6 True
P3     1        Player       9        9 True
P4     1        Player       8        8 True
P5     1        Player       3        3 True
P6     1        Player      14       14 True
P7     1        Player       4        4 True
P8     1        Player      12       12 True
T1     1 Participation       7        7 True
T2     1 Participation      11       11 True
T3     1 Participation       8        8 True
T4     1 Participation       3        3 True
M1     1         Match       5        5 True
M2     1         Match       4        4 True
M3     1         Match      10       10 True
M4     1         Match       3        3 True
M5     1         Match      17       17 True
F1     1    Federation       3        3 True
C1     1          Club      22       22 True
S1     1       Stadium       1        1 True
X1     4         Match       3        3 True
insiemi pe

## E1 — Classifiche delle quattro strategie
Per ogni domanda, i primi 10 risultati di ogni strategia:
- **solo semantica**: il vettore della domanda intera sull'indice della classe, nessun filtro;
- **indice prima (50 / 100 / 500)**: il vettore della sola parte descrittiva (`testo_semantico`) sull'indice, poi il filtro del grafo sui candidati;
- **grafo prima**: il filtro del grafo restringe i nodi, poi il coseno esatto con la parte descrittiva;
- **grafo prima (domanda intera)**: Come grafo prima, ma con il vettore della domanda intera, come in solo semantica. Il confronto con solo semantica misura il solo effetto del filtro.


In [3]:
# [Cella 3] E1: classifiche (10 risultati per strategia) e tempi
def vettori_domanda(d):
    """(vettore per la semantica pura, vettore per le strategie con il grafo, nodo da escludere)."""
    if d.get("vettore_da"):
        uri = BASE + d["vettore_da"]
        v = g.leggi("MATCH (n:Resource {uri: $u}) RETURN n.embedding AS e", u=uri)["e"].iloc[0]
        return list(v), list(v), uri
    return frase(minilm, d["domanda"]), frase(minilm, d["testo_semantico"]), ""

# Le strategie confrontate: semantica pura, indice prima (3 varianti), grafo prima, controllo
STRATEGIE = ["solo semantica", "indice prima 50", "indice prima 100", "indice prima 500", "grafo prima",
             "grafo prima (domanda intera)"]
classifiche = []
for d in E1:
    v_tutto, v_parte, escludi = vettori_domanda(d)
    for s in STRATEGIE:
        # Si misura anche il tempo di ogni ricerca
        t0 = time.time()
        if s == "solo semantica":
            r = val.solo_semantica(g, d["classe"], v_tutto, escludi=escludi)
        elif s.startswith("indice prima"):
            r = val.indice_prima(g, d["classe"], v_parte, d["filtro"], int(s.split()[-1]), escludi=escludi)
        else:  # il controllo usa la domanda intera, come solo semantica
            v = v_tutto if s == "grafo prima (domanda intera)" else v_parte
            r = val.grafo_prima(g, d["classe"], v, d["filtro"], escludi=escludi)
        ms = (time.time() - t0) * 1000
        per = pertinenti.get(d["id"])
        for i, x in enumerate(r.itertuples(), 1):
            classifiche.append({"id": d["id"], "caso": d["caso"], "strategia": s, "posizione": i, "uri": x.uri,
                                "nome": x.nome, "score": round(x.score, 4), "ms": round(ms),
                                "pertinente": None if per is None else x.uri in per})
        # Domanda senza risultati: si registra comunque, per contare le domande vuote
        if r.empty:
            classifiche.append({"id": d["id"], "caso": d["caso"], "strategia": s, "posizione": None, "uri": None,
                                "nome": None, "score": None, "ms": round(ms), "pertinente": None})
classifiche = pd.DataFrame(classifiche)
classifiche.to_csv(RIS / "e1_classifiche.csv", index=False, encoding="utf-8")
print("righe:", len(classifiche), "| salvato:", RIS / "e1_classifiche.csv")
print("tempo medio per strategia (ms):", classifiche.groupby("strategia")["ms"].mean().round().to_dict())
esempio = classifiche[(classifiche["id"] == "P1")].dropna(subset=["uri"])
print(esempio.pivot(index="posizione", columns="strategia", values="nome")[STRATEGIE].to_string())

righe: 1132 | salvato: <progetto>\data\valutazione\risultati\e1_classifiche.csv
tempo medio per strategia (ms): {'grafo prima': 95.0, 'grafo prima (domanda intera)': 60.0, 'indice prima 100': 25.0, 'indice prima 50': 25.0, 'indice prima 500': 22.0, 'solo semantica': 42.0}
strategia   solo semantica indice prima 50 indice prima 100 indice prima 500     grafo prima grafo prima (domanda intera)
posizione                                                                                                                
1.0                Germano         Ricardo          Ricardo          Ricardo         Ricardo                   Vítor Baía
2.0         Carlos Germano    Rogério Ceni     Rogério Ceni     Rogério Ceni    Rogério Ceni              Pedro Contreras
3.0           Paulo Sérgio      Vítor Baía       Vítor Baía       Vítor Baía      Vítor Baía                       Marcos
4.0                Pedrosa         Ricardo          Ricardo          Ricardo         Ricardo                 Rogério 

## E1 — Metriche
Sulle 21 domande con insieme pertinente: **P@5** (quota dei primi 5 giusti), **R@10** (quota dei giusti tra i primi 10), **nDCG@10** (premia i giusti in cima; 1 = ordine perfetto), **MRR** (media di 1/posizione del primo giusto), **vuote** .

In [4]:
# [Cella 4] E1: metriche per domanda e per strategia
righe = []
# Metriche solo per le domande con insieme pertinente (non X2–X4)
for d in E1:
    if d["id"] not in pertinenti:
        continue
    for s in STRATEGIE:
        c = classifiche[(classifiche["id"] == d["id"]) & (classifiche["strategia"] == s)].dropna(subset=["uri"])
        righe.append({"id": d["id"], "caso": d["caso"], "classe": d["classe"], "strategia": s,
                      "pertinenti": len(pertinenti[d["id"]]), **val.metriche(c["uri"].tolist(), pertinenti[d["id"]])})
m2 = pd.DataFrame(righe)
m2.to_csv(RIS / "e1_metriche.csv", index=False, encoding="utf-8")
# Media sulle domande, per strategia
riepilogo = (m2.groupby("strategia")[["P@5", "R@10", "nDCG@10", "RR", "vuota"]].mean()
             .rename(columns={"RR": "MRR", "vuota": "quota vuote"}).loc[STRATEGIE].round(3))
print("Media sulle domande con insieme pertinente:")
print(riepilogo.to_string())
print("\nnDCG@10 per domanda:")
print(m2.pivot(index="id", columns="strategia", values="nDCG@10")[STRATEGIE].round(2).to_string())

Media sulle domande con insieme pertinente:
                                P@5   R@10  nDCG@10    MRR  quota vuote
strategia                                                              
solo semantica                0.324  0.423    0.450  0.673        0.000
indice prima 50               0.238  0.259    0.299  0.427        0.333
indice prima 100              0.286  0.350    0.382  0.528        0.143
indice prima 500              0.352  0.450    0.461  0.580        0.048
grafo prima                   0.410  0.485    0.504  0.643        0.000
grafo prima (domanda intera)  0.438  0.532    0.580  0.788        0.000

nDCG@10 per domanda:
strategia  solo semantica  indice prima 50  indice prima 100  indice prima 500  grafo prima  grafo prima (domanda intera)
id                                                                                                                       
C1                   0.52             0.47              0.47              0.47         0.47                       

## E2a — CLIP riconosce la persona?
*"a photo of {nome}"* su tutte le immagini; pertinenti = le immagini collegate al giocatore. **P@5** per 15 giocatori famosi e 15 presi a caso.

In [5]:
# [Cella 5] E2a: riconoscimento della persona, P@5
righe = []
for gruppo in ("famosi", "meno_noti"):
    for pid in E2["a_persona"][gruppo]:
        uri = BASE + "player/" + pid
        nome = g.leggi("MATCH (p:Resource {uri: $u}) RETURN p.label AS nome", u=uri)["nome"].iloc[0]
        # Pertinenti: le immagini collegate al giocatore con depicts
        sue = set(g.leggi("MATCH (:Resource {uri: $u})<-[:depicts]-(i:Image) RETURN i.uri AS uri", u=uri)["uri"])
        # 'a photo of {nome}' → le 5 immagini più vicine secondo CLIP
        r = val.immagini_indice(g, frase(clip, E2["a_persona"]["testo"].format(nome=nome)), k=5)
        righe.append({"gruppo": gruppo, "giocatore": nome, "immagini": len(sue),
                      "P@5": val.precisione(r["uri"].tolist(), sue, 5),
                      "primi_5": ", ".join(r["giocatore"])})
e2a = pd.DataFrame(righe)
e2a.to_csv(RIS / "e2a_persona.csv", index=False, encoding="utf-8")
print(e2a.groupby("gruppo")["P@5"].agg(["mean", "count"]).round(3).to_string())
print(e2a.drop(columns="primi_5").to_string(index=False))

            mean  count
gruppo                 
famosi     0.707     15
meno_noti  0.133     15
   gruppo          giocatore  immagini  P@5
   famosi       Lionel Messi        50  0.8
   famosi  Cristiano Ronaldo        50  0.4
   famosi             Neymar        50  0.4
   famosi      Kylian Mbappé        50  1.0
   famosi        Luka Modrić        50  1.0
   famosi         Harry Kane        50  0.4
   famosi    Kevin De Bruyne        50  1.0
   famosi Robert Lewandowski        50  0.8
   famosi  Antoine Griezmann        50  1.0
   famosi      Heung-min Son        50  0.2
   famosi    Virgil van Dijk        50  1.0
   famosi      Thomas Müller        50  0.8
   famosi        Luis Suárez        50  0.0
   famosi       Manuel Neuer        50  1.0
   famosi        Gareth Bale        50  0.8
meno_noti     Thomas Delaney        50  0.8
meno_noti  Raphaël Guerreiro        50  0.0
meno_noti       Wahbi Khazri        50  0.0
meno_noti  Bilal El Khannous        50  0.0
meno_noti       Ismaïla 

## E2b — Concetti visivi con etichetta ricavata dal grafo
Il grafo dice chi è portiere e chi gioca in quale nazionale: le immagini di quei giocatori fanno da pertinenti (etichetta rumorosa: la foto di un portiere in borghese conta come giusta). La **base casuale** è la quota di pertinenti tra le immagini di partenza: è la P@10 che si otterrebbe pescando a caso.

In [6]:
# [Cella 6] E2b: P@10 contro la base casuale
righe = []
for c in E2["b_concetti"]:
    # Pertinenti ricavati dal grafo (portieri, convocati di una nazionale) e immagini di partenza
    per, tutte = val.immagini_pertinenti(g, c["pertinenti"], c["filtro"])
    v = frase(clip, c["testo"])
    r = val.immagini_indice(g, v, k=10) if c["filtro"] is None else val.immagini_squadra(g, v, c["filtro"].split(":", 1)[1], k=10)
    p10 = val.precisione(r["uri"].tolist(), per, 10)
    # Base casuale: la precisione attesa pescando a caso
    base = len(per) / len(tutte)
    righe.append({"id": c["id"], "testo": c["testo"], "modo": "tutte le immagini" if c["filtro"] is None else c["filtro"],
                  "partenza": len(tutte), "pertinenti": len(per), "base casuale": round(base, 3), "P@10": p10,
                  "volte la base": round(p10 / base, 1) if base else None})
e2b = pd.DataFrame(righe)
e2b.to_csv(RIS / "e2b_concetti.csv", index=False, encoding="utf-8")
print(e2b.to_string(index=False))

id                                                                    testo              modo  partenza  pertinenti  base casuale  P@10  volte la base
B1                                                  a photo of a goalkeeper tutte le immagini     41460        4950         0.119   1.0            8.4
B2                     a football player wearing the yellow shirt of Brazil tutte le immagini     41460        1300         0.031   0.7           22.3
B3            a football player wearing the orange shirt of the Netherlands tutte le immagini     41460        1300         0.031   0.9           28.7
B4 a football player in the light blue and white striped shirt of Argentina tutte le immagini     41460        1300         0.031   0.9           28.7
B5                                                  a photo of a goalkeeper    squadra:Brazil      1300         150         0.115   0.9            7.8


## Righe da giudicare a mano
- **E1, domande senza risposta esatta** (X2–X4): l'unione dei primi 10 di solo semantica, indice prima 100 e grafo prima, con il criterio di giudizio della suite;
- **E2c**: i primi 10 di cinque concetti visivi su tutte le immagini;
- **E2d**: per 20 giocatori, l'immagine della scheda (non duplicata, `labelConfidence` più alta) e un'immagine a caso della stessa cartella.

In [7]:
# [Cella 7] Scrive da_giudicare.csv (colonna "pertinente" vuota, da compilare)
righe = []
# E1 X2–X4: unione dei primi 10 di tre strategie, da giudicare a mano
for d in E1:
    if d["id"] in pertinenti:
        continue
    c = classifiche[(classifiche["id"] == d["id"]) & classifiche["strategia"].isin(["solo semantica", "indice prima 100", "grafo prima"])]
    for x in c.dropna(subset=["uri"]).drop_duplicates("uri").itertuples():
        righe.append({"compito": "E1", "id": d["id"], "criterio": d["criterio"], "uri": x.uri, "nome": x.nome, "percorso": None})
# E2c: i primi 10 di ogni concetto visivo
for c in E2["c_giudizi"]:
    for x in val.immagini_indice(g, frase(clip, c["testo"]), k=10).itertuples():
        righe.append({"compito": "E2c", "id": c["id"], "criterio": c["testo"], "uri": x.uri, "nome": x.giocatore, "percorso": x.percorso})
# E2d: immagine della scheda contro un'immagine a caso (seme fisso: ripetibile)
casuale = random.Random(E2["d_scheda"]["seme"])
for pid in E2["d_scheda"]["giocatori"]:
    imm = g.leggi("""MATCH (p:Resource {uri: $u})<-[:depicts]-(i:Image)
                     RETURN i.uri AS uri, i.filePath AS percorso, p.label AS nome
                     ORDER BY i.hasDuplicateContent, coalesce(i.labelConfidence, -1.0) DESC, i.filePath""",
                  u=BASE + "player/" + pid)
    for tipo, x in (("scheda", imm.iloc[0]), ("a caso", imm.iloc[casuale.randrange(1, len(imm))])):
        righe.append({"compito": "E2d", "id": f"{pid} {tipo}", "criterio": f"ritrae {x['nome']} (da solo o riconoscibile)?",
                      "uri": x["uri"], "nome": x["nome"], "percorso": x["percorso"]})
da_giudicare = pd.DataFrame(righe).assign(pertinente=None, nota=None)
da_giudicare.to_csv(RIS / "da_giudicare.csv", index=False, encoding="utf-8")
print("da giudicare:", da_giudicare.groupby("compito").size().to_dict(), "| salvato:", RIS / "da_giudicare.csv")

da giudicare: {'E1': 52, 'E2c': 50, 'E2d': 40} | salvato: <progetto>\data\valutazione\risultati\da_giudicare.csv


## Metriche dei giudizi a mano
Per le domande X2–X4 non c'è un insieme pertinente completo, quindi solo **P@5** e **P@10**, non il richiamo.

In [5]:
# [Cella 8] Metriche dai giudizi a mano (E1 X2–X4, E2c, E2d)
file = RIS / "giudizi.csv"
if not file.exists():
    print("giudizi.csv non c'è ancora: compila la colonna pertinente di da_giudicare.csv e salvalo come giudizi.csv.")
else:
    # Giudizi a mano: pertinente = 1 oppure 0
    giu = pd.read_csv(file)
    # Per ogni caso, l'insieme degli URI giudicati pertinenti
    giusti = {(r.compito, r.id): set() for r in giu.itertuples()}
    for r in giu[giu["pertinente"] == 1].itertuples():
        giusti[(r.compito, r.id)].add(r.uri)
    # E1: precisione delle tre strategie con la pertinenza giudicata
    righe = []
    for d in E1:
        if d["id"] in pertinenti:
            continue
        for s in ["solo semantica", "indice prima 100", "grafo prima"]:
            lista = classifiche[(classifiche["id"] == d["id"]) & (classifiche["strategia"] == s)].dropna(subset=["uri"])["uri"].tolist()
            righe.append({"id": d["id"], "caso": d["caso"], "strategia": s,
                          "P@5": val.precisione(lista, giusti[("E1", d["id"])], 5),
                          "P@10": val.precisione(lista, giusti[("E1", d["id"])], 10)})
    print("E1, domande giudicate a mano:")
    print(pd.DataFrame(righe).to_string(index=False))
    # E2c: quota di pertinenti tra i primi 10 di ogni concetto
    e2c = giu[giu["compito"] == "E2c"].groupby(["id", "criterio"])["pertinente"].mean().rename("P@10").reset_index()
    print("\nE2c, concetti visivi:")
    print(e2c.to_string(index=False), f"\nmedia P@10: {e2c['P@10'].mean():.2f}")
    # E2d: P@1 dell'immagine della scheda contro un'immagine a caso
    e2d = giu[giu["compito"] == "E2d"].assign(tipo=lambda x: x["id"].str.split(" ", n=1).str[1])
    print("\nE2d, P@1:", e2d.groupby("tipo")["pertinente"].mean().round(2).to_dict())

E1, domande giudicate a mano:
id  caso        strategia  P@5  P@10
X2     2   solo semantica  0.2   0.1
X2     2 indice prima 100  0.2   0.1
X2     2      grafo prima  0.2   0.1
X3     3   solo semantica  0.2   0.2
X3     3 indice prima 100  0.4   0.3
X3     3      grafo prima  0.4   0.3
X4     2   solo semantica  0.2   0.2
X4     2 indice prima 100  0.0   0.0
X4     2      grafo prima  0.6   0.3

E2c, concetti visivi:
id                                   criterio  P@10
V1       a football player celebrating a goal   0.7
V2           a team photo of a football squad   0.7
V3       a goalkeeper diving to save the ball   0.7
V4         a football player holding a trophy   0.9
V5 a close-up portrait of a footballer's face   1.0 
media P@10: 0.80

E2d, P@1: {'a caso': 0.9, 'scheda': 1.0}
